In [6]:
import pickle
from torch.utils.data import DataLoader
from data_utils import latent_MH_collate_fn

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
dataset_path = 'data/latent_datasets/gjt_CA.pickle'

with open(dataset_path, 'rb') as f:
    dataset = pickle.load(f)

In [8]:
print(len(dataset))
print(dataset[0].keys())
print(dataset[0]['latent'].shape)

1857
dict_keys(['harmony_ids', 'attention_mask', 'pianoroll', 'time_signature', 'h_density_complexity', 'latent'])
(512,)


In [9]:
val_loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=latent_MH_collate_fn)

In [10]:
batch = next(iter(val_loader))

In [11]:
print(batch["harmony_ids"].shape)

torch.Size([4, 80])


In [12]:
from train_utils import make_mixed_batch

In [13]:
mixed_batch_1 = make_mixed_batch(batch, "latent")
mixed_batch_2 = make_mixed_batch(mixed_batch_1, "latent")

In [30]:
harmony_gt = batch["harmony_ids"]
foreign_ids = mixed_batch_1['harmony_ids']

In [18]:
import torch
from GridMLM_tokenizers import CSGridMLMTokenizer

In [23]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [27]:
print(len(tokenizer.vocab))

355


In [28]:
logits = torch.rand((4,len(tokenizer.vocab)))

In [40]:
import torch.nn.functional as F

def compute_unique_logit_activations(harmony_gt, foreign_ids, logits):
    """
    harmony_gt : [B, L] token ids
    foreign_ids: [B, L] token ids
    logits     : [B, D]

    Returns
    -------
    home_unique_logits_activations    : [B]
    foreign_unique_logits_activations : [B]
    """

    B, D = logits.shape

    # Convert logits to probabilities
    probs = F.softmax(logits, dim=1)   # [B, D]

    # Create vocabulary masks
    home_mask = torch.zeros(B, D, dtype=torch.bool, device=logits.device)
    foreign_mask = torch.zeros(B, D, dtype=torch.bool, device=logits.device)

    home_mask.scatter_(1, harmony_gt, True)
    foreign_mask.scatter_(1, foreign_ids, True)

    # Unique tokens
    home_unique_mask = home_mask & (~foreign_mask)
    foreign_unique_mask = foreign_mask & (~home_mask)
    print(home_unique_mask.sum())
    print(foreign_unique_mask.sum())

    # Sum probabilities
    home_unique_sum = (probs * home_unique_mask).sum(dim=1)
    foreign_unique_sum = (probs * foreign_unique_mask).sum(dim=1)

    # Normalize by total probability mass (softmax sums to 1, but kept explicit)
    total_prob = probs.sum(dim=1)

    home_unique_logits_activations = home_unique_sum / total_prob
    foreign_unique_logits_activations = foreign_unique_sum / total_prob

    return home_unique_logits_activations, foreign_unique_logits_activations

In [41]:
home_unique_logits_activations, foreign_unique_logits_activations = compute_unique_logit_activations(harmony_gt, foreign_ids, logits)

tensor(43)
tensor(43)


In [32]:
print(home_unique_logits_activations)
print(foreign_unique_logits_activations)

tensor([0.0427, 0.0139, 0.0344, 0.0281])
tensor([0.0179, 0.0487, 0.0282, 0.0327])
